# Improving Reasoning

Replicates **"Improving Reasoning Performance in Large Language Models via Representation Engineering"** ([arXiv:2504.19483](https://arxiv.org/abs/2504.19483)) on Mistral-7B-Instruct-v0.1, end to end in one engine:

1. **Construction** — contrastive pairs (`reasoning_pairs.json`) share the same GSM8K-style questions, answered with sound vs flawed reasoning; the difference of means of the last-token hidden states gives a control vector (`reason.gguf`).
2. **Steering** — adding it at layers 16–19 on every token; on this GSM8K problem the baseline reasons incorrectly while the steered model solves it.

Uses the current EasySteer steering API (`SteeringSpec` / `VectorSpec` / `ApplySpec`).

**Execution note:** Outputs are cleared after the API migration. Run the notebook from top to bottom to obtain results for your environment.


In [ ]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

from vllm import LLM, SamplingParams
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec

MODEL = os.environ.get("EASYSTEER_MODEL", "mistralai/Mistral-7B-Instruct-v0.1")  # mistralai/Mistral-7B-Instruct-v0.1

# One engine serves both construction (capture) and steering.
llm = LLM(
    model=MODEL,
    tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1")),
    enable_steer_vector=True,
    steer_algorithms=["direct"],
)
tokenizer = llm.get_tokenizer()

## Vector construction

In [ ]:
import json

with open("reasoning_pairs.json", encoding="utf-8") as f:
    pairs = json.load(f)

# Same question, sound vs. flawed reasoning, in Mistral [INST] format.
formatted_positive = [f"[INST] {p['question']} [/INST] {p['sound']}" for p in pairs]
formatted_negative = [f"[INST] {p['question']} [/INST] {p['flawed']}" for p in pairs]

In [ ]:
from easysteer.capture import capture_batches
from vllm.capture import SelectSpec

# Only the last prompt row feeds the extractor, so select it at the source.
batches = capture_batches(
    llm,
    formatted_positive + formatted_negative,
    select=SelectSpec(prompt_positions=[-1]),
    steering=False,
)

In [ ]:
from easysteer.extraction import extract

# Consume each batch once: mean(sound) minus mean(flawed).
labels = [True] * len(pairs) + [False] * len(pairs)
control_vector = extract(
    batches,
    labels,
    method="diffmean",
    token_pos=-1,
    normalize=True,
)
control_vector.export_gguf("reason.gguf")


## Steering

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Solve the problem: A robe takes 2 bolts of blue fiber and half that much white fiber. How many bolts in total does it take?",
    },
]
prompt_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, return_dict=False, add_generation_prompt=True,
)
prompt = {"prompt_token_ids": prompt_ids}
params = SamplingParams(temperature=0, max_tokens=256, skip_special_tokens=False)

# The unsteered model gets this problem wrong.
baseline = llm.generate(prompt, params, use_tqdm=False, steering=False)
print("=====Baseline=====")
print(baseline[0].outputs[0].text)

In [ ]:
# Add the reasoning direction at layers 16-19, on every prompt and
# generated token.
steering = SteeringSpec(vectors=[
    VectorSpec(
        source="reason.gguf",
        scale=1.0,
        layers=list(range(16, 20)),
        apply=ApplySpec(prompt="all", generation="all"),
    ),
])

steered = llm.generate(prompt, params, steering=steering, use_tqdm=False)
print("=====Reasoning Steered=====")
print(steered[0].outputs[0].text)